# §12.1.5 — 가변 길이 시퀀스의 패딩과 마스킹 구현

> 딥러닝 교재 · 3부 12장 1절 5항 (🐍)
> 선행: §12.1.1(상태 갱신식) · §12.1.5(마스킹의 두 자리) · §12.1.7(조용한 버그)

## 이 노트북이 답하는 질문

1. **손실 마스킹을 빠뜨리면** 손실은 어떻게 보이고 성능은 어떻게 되는가?
2. **상태 마스킹을 빠뜨리면** 어떤 시퀀스가 특히 다치는가?
3. 두 마스킹은 왜 **독립적으로** 필요한가?

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 50초).
실무에서 가장 자주 틀리는 지점을 일부러 틀려 보는 노트북이다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 마스킹이 옳아야만 풀리는 설계

길이 $L\in[4,16]$의 시퀀스(표준정규 값), 최대 길이 $T=16$으로 패딩.
**패딩 값은 0이 아니라 잡음**으로 채운다 — 실제 배치의 패딩 영역에 무엇이 있을지 가정하지 않는 것이 안전한 설계다.

| 출력 | 정의 | 다치는 마스킹 |
|---|---|---|
| 시각별 예측 | $y_t=\operatorname{sign}(\sum_{s\le t}x_s)$ (유효 구간만 채점) | 손실 마스킹 |
| 최종 요약 예측 | 시퀀스 전체 합의 부호를 **마지막 상태** $h_T$에서 판독 | 상태 마스킹 |

In [ ]:
T = 16; M = 16
def make_data(n, rn):
    L = rn.integers(4, T+1, n)
    X = rn.standard_normal((n, T)) * 1.0          # 패딩 영역에도 잡음이 남는다
    mask = (np.arange(T)[None, :] < L[:, None]).astype(float)
    Xv = X * mask                                  # 유효 구간의 값
    csum = np.cumsum(Xv, axis=1)
    y_step = np.sign(csum + 1e-9)                  # 시각별 레이블 (유효 구간만 쓴다)
    y_seq = np.sign(csum[np.arange(n), L-1] + 1e-9)
    return X, mask, y_step, y_seq, L

Xte, Mte, Yte_step, Yte_seq, Lte = make_data(2000, np.random.default_rng(SEED+9))
print("시험:", Xte.shape, " 길이 분포:", np.bincount(Lte)[4:])

---
## 2. 순환망과 BPTT — 마스킹을 선택할 수 있게 구현

In [ ]:
def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

class RNN:
    def __init__(self, rn, m=M):
        self.Wh = rn.standard_normal((m, m)) * (0.9/np.sqrt(m))
        self.Wx = rn.standard_normal((1, m)) * 1.0
        self.b = np.zeros(m)
        self.U = rn.standard_normal(m) / np.sqrt(m); self.c = np.zeros(1)   # 시각별 판독
        self.V = rn.standard_normal(m) / np.sqrt(m); self.d = np.zeros(1)   # 최종 판독
        self.params = [self.Wh, self.Wx, self.b, self.U, self.c, self.V, self.d]
    def forward(self, X, mask, state_mask=True):
        B = X.shape[0]; m = self.b.size
        H = np.zeros((B, T+1, m)); Z = np.zeros((B, T, m))
        for t in range(T):
            Z[:, t] = H[:, t] @ self.Wh + X[:, t:t+1] @ self.Wx + self.b
            h_new = np.tanh(Z[:, t])
            if state_mask:
                mt = mask[:, t:t+1]
                H[:, t+1] = mt*h_new + (1-mt)*H[:, t]
            else:
                H[:, t+1] = h_new
        logit_step = H[:, 1:] @ self.U + self.c        # (B,T)
        logit_seq = H[:, T] @ self.V + self.d          # (B,)
        self.cache = (X, mask, H, Z, state_mask)
        return logit_step, logit_seq
    def backward(self, dstep, dseq):
        X, mask, H, Z, state_mask = self.cache
        B = X.shape[0]
        gWh = np.zeros_like(self.Wh); gWx = np.zeros_like(self.Wx); gb = np.zeros_like(self.b)
        gU = np.einsum('btm,bt->m', H[:, 1:], dstep); gc = np.array([dstep.sum()])
        gV = H[:, T].T @ dseq; gd = np.array([dseq.sum()])
        delta = np.outer(dseq, self.V)                 # dL/dH[:,T]
        for t in range(T-1, -1, -1):
            delta = delta + np.outer(dstep[:, t], self.U)
            if state_mask:
                mt = mask[:, t:t+1]
                dh_new = mt * delta
                carry = (1-mt) * delta
            else:
                dh_new = delta; carry = 0.
            dz = dh_new * (1 - np.tanh(Z[:, t])**2)
            gWh += H[:, t].T @ dz
            gWx += (X[:, t:t+1]).T @ dz
            gb += dz.sum(axis=0)
            delta = dz @ self.Wh.T + carry
        return [gWh, gWx, gb, gU, gc, gV, gd]

def adam(p, g, m, v, t, lr=4e-3):
    m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
    p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)

def train(loss_mask=True, state_mask=True, steps=None, seed=0, track=None):
    steps = steps or (200 if FAST else 450)
    net = RNN(np.random.default_rng(seed))
    Xtr, Mtr, Ytr_step, Ytr_seq, Ltr = make_data(6000, np.random.default_rng(SEED+seed))
    ms = [np.zeros_like(p) for p in net.params]; vs = [np.zeros_like(p) for p in net.params]
    rb = np.random.default_rng(100+seed); B = 128
    hist = {'step': [], 'loss': [], 'acc': []}
    for t in range(1, steps+1):
        idx = rb.integers(0, len(Ltr), B)
        ls, lq = net.forward(Xtr[idx], Mtr[idx], state_mask=state_mask)
        p_step = sigmoid(ls); p_seq = sigmoid(lq)
        y01 = (Ytr_step[idx] > 0).astype(float); q01 = (Ytr_seq[idx] > 0).astype(float)
        if loss_mask:
            w = Mtr[idx]/Mtr[idx].sum()
        else:
            # 흔한 실수의 재현: 패딩 위치도 손실에 포함하고, 패딩 레이블은 관례적으로 0(음수 클래스)
            y01 = y01 * Mtr[idx]
            w = np.ones_like(Mtr[idx])/Mtr[idx].size
        dstep = (p_step - y01) * w
        dseq = (p_seq - q01) / B
        gs = net.backward(dstep, dseq)
        for pp, g, m_, v_ in zip(net.params, gs, ms, vs):
            adam(pp, g, m_, v_, t)
        if track is not None and (t % 15 == 0 or t == 1):
            ls_e, lq_e = net.forward(Xte[:800], Mte[:800], state_mask=state_mask)
            pe = sigmoid(ls_e); ye = (Yte_step[:800] > 0)
            eps = 1e-9
            Me_ = Mte[:800]
            # 유효 위치만의 손실 (진짜 지표)
            ye_eff = ye.astype(float)
            nll_pt = -(np.where(ye, np.log(pe+eps), np.log(1-pe+eps)))
            nll_valid = (nll_pt * Me_).sum()/Me_.sum()
            # 보고되는 손실: 마스킹 누락 시 패딩(레이블 0) 포함 평균
            ye_rep = ye_eff * Me_
            nll_rep_pt = -(ye_rep*np.log(pe+eps) + (1-ye_rep)*np.log(1-pe+eps))
            nll_reported = nll_rep_pt.mean() if not loss_mask else nll_valid
            acc = (((ls_e > 0) == ye) * Me_).sum()/Me_.sum()
            hist['step'].append(t); hist['loss'].append((nll_valid, nll_reported)); hist['acc'].append(acc)
    return net, hist

def eval_all(net, state_mask=True):
    ls, lq = net.forward(Xte, Mte, state_mask=state_mask)
    acc_step = (((ls > 0) == (Yte_step > 0)) * Mte).sum()/Mte.sum()
    acc_seq_by_len = {}
    ok = (lq > 0) == (Yte_seq > 0)
    for L0 in range(4, T+1):
        sel = Lte == L0
        acc_seq_by_len[L0] = ok[sel].mean()
    return acc_step, ok.mean(), acc_seq_by_len

---
## 3. 세 가지 방식으로 학습

In [ ]:
net_ok, hist_ok = train(loss_mask=True, state_mask=True, seed=1, track=True)
net_nl, hist_nl = train(loss_mask=False, state_mask=True, seed=1, track=True)
net_ns, hist_ns = train(loss_mask=True, state_mask=False, seed=1, track=True)

for name, net, sm in [('올바른 마스킹', net_ok, True), ('손실 마스킹 누락', net_nl, True), ('상태 마스킹 누락', net_ns, False)]:
    a1, a2, _ = eval_all(net, state_mask=sm)
    print(f"{name}: 시각별(유효) {a1:.3f}  최종 요약 {a2:.3f}")

---
## 4. 교재 그림 — fig_12_1_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 배치와 마스크
ax = axes[0]
Xs, Ms_, _, _, Ls_ = make_data(8, np.random.default_rng(3))
show_img = np.where(Ms_ > 0, Xs, np.nan)
imv = ax.imshow(Xs, cmap='gray', aspect='auto'); ax.grid(False)
for i, L0 in enumerate(Ls_):
    ax.plot([L0-0.5, L0-0.5], [i-0.5, i+0.5], color=CB[4], lw=2)
ax.set_xlabel(lab('시각 $t$', 'time $t$')); ax.set_ylabel(lab('배치 내 시퀀스', 'sequence'))
ax.set_title(lab('(a) 패딩된 배치 — 주황 선 오른쪽이 패딩(잡음)', '(a) padded batch'), fontsize=10)

# (b) 학습 곡선 (유효 정확도)
ax = axes[1]
for hist, name, c in [(hist_ok, lab('올바른 마스킹', 'correct'), CB[5]),
                      (hist_nl, lab('손실 마스킹 누락', 'no loss mask'), CB[4]),
                      (hist_ns, lab('상태 마스킹 누락', 'no state mask'), CB[1])]:
    ax.plot(hist['step'], hist['acc'], '-', color=c, lw=1.4, label=name)
ax.set_xlabel(lab('학습 걸음', 'step')); ax.set_ylabel(lab('유효 위치 정확도', 'valid-position accuracy'))
ax.set_title(lab('(b) 유효 위치 성능', '(b) accuracy on valid positions'), fontsize=10)
ax.legend(fontsize=8)

# (c) 손실 마스킹 누락 — 보고되는 손실 vs 진짜(유효) 손실
ax = axes[2]
v_ok = [l[0] for l in hist_ok['loss']]
v_nl = [l[0] for l in hist_nl['loss']]
r_nl = [l[1] for l in hist_nl['loss']]
ax.plot(hist_nl['step'], r_nl, '-', color=CB[4], lw=1.5,
        label=lab('마스킹 누락 모델: 보고되는 손실', 'reported loss (no mask)'))
ax.plot(hist_nl['step'], v_nl, '--', color=CB[4], lw=1.5,
        label=lab('같은 모델: 유효 위치만의 손실', 'its true valid loss'))
ax.plot(hist_ok['step'], v_ok, '-', color=CB[5], lw=1.2,
        label=lab('올바른 모델: 유효 손실', 'correct model valid loss'))
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('시각별 NLL', 'per-step NLL'))
ax.set_title(lab('(c) 보고 손실이 진짜 손실을 가린다', '(c) reported vs true loss'), fontsize=10)
ax.legend(fontsize=7)

# (d) 상태 마스킹 누락 — 길이별 최종 요약 정확도
ax = axes[3]
_, _, by_ok = eval_all(net_ok, True)
_, _, by_ns = eval_all(net_ns, False)
Ls = sorted(by_ok)
ax.plot(Ls, [by_ok[L0] for L0 in Ls], 'o-', color=CB[5], ms=4, label=lab('올바른 마스킹', 'correct'))
ax.plot(Ls, [by_ns[L0] for L0 in Ls], 's-', color=CB[1], ms=4, label=lab('상태 마스킹 누락', 'no state mask'))
ax.set_xlabel(lab('실제 시퀀스 길이 $L$', 'true length $L$'))
ax.set_ylabel(lab('최종 요약 정확도', 'final-readout accuracy'))
ax.set_title(lab('(d) 짧은 시퀀스일수록 패딩에 씻긴다', '(d) short sequences suffer most'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_12_1_5')
plt.show()

> ### 읽는 법
>
> (b, c) 손실 마스킹을 빠뜨린 모델은 패딩 위치의 관례 레이블(0)까지 학습 목표에 넣는다.
> 문제는 상태가 얼어 있는 패딩 위치와 마지막 유효 위치가 **같은 상태를 공유**한다는 것 —
> 같은 입력에 다른 답을 요구받으니 유효 위치의 성능이 끌려 내려간다.
> 그런데 (c)처럼 **보고되는 손실**(패딩 포함 평균)은 진짜(유효 위치) 손실보다 낮게 찍힌다.
> 나빠진 것은 가려지고 좋아 보이는 숫자만 남는, §12.1.7의 조용한 버그다.
> (d) 상태 마스킹을 빠뜨리면 **마지막 상태가 패딩 잡음에 씻긴다.** 패딩이 긴, 즉 짧은 시퀀스일수록
> 심하게 다치는 계단 모양이 지문이다.
> 두 마스킹은 서로를 대신하지 못한다 — 다치는 출력이 다르기 때문이다.

---
## 5. 자기 점검

1. (c)에서 패딩 위치의 레이블을 무엇으로 두었는지 확인하라. 그 선택이 "패딩이 맞히기 쉽다"를 어떻게 만들었는가?
2. 패딩을 잡음이 아니라 0으로 채우면 (d)의 계단이 완만해진다. 왜인가? 그래도 상태 마스킹이 필요한 이유는?
3. 시각별 판독만 있고 최종 판독이 없는 모델이라면 상태 마스킹 누락은 무엇을 다치게 하는가?
4. 프레임워크의 packed sequence는 두 마스킹 중 무엇을 대신해 주는가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| 패딩 값 | 1절 | 잡음 | 0으로 바꿔 (d) 재관찰 |
| `T`, 길이 범위 | 1절 | 16, [4,16] | 길이 분산이 클수록 함정이 커진다 |
| `steps` | 2절 | 450 | 학습 길이 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")